# NexaLang Transformer — Treinamento no Google Colab

Treina um modelo Transformer pre-norm (mesma arquitetura do NexaLang v7+) usando **PyTorch + CUDA**.

**Modelos disponíveis:**
| Tamanho | Params | V | D | H | N | VRAM |
|---------|--------|-------|------|---|----|------|
| small   | ~33M   | 1024  | 512  | 8 | 10 | ~2GB |
| medium  | ~150M  | 16384 | 768  | 12| 12 | ~8GB |
| large   | ~500M  | 32000 | 1024 | 16| 24 | ~18GB|

**Instruções:**
1. Vá em `Runtime > Change runtime type > GPU` (T4 grátis ou A100 Pro)
2. Execute cada célula na ordem
3. Na célula de configuração, escolha o tamanho do modelo
4. Os pesos são salvos no formato binário do NexaLang
5. Baixe os arquivos no final para usar com `chat_v8`

In [ ]:
#@title 1. Setup & GPU Check
!pip install -q sentencepiece datasets tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import struct, os, time, math, gc, random
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}')
    print(f'VRAM: {vram_gb:.1f} GB')
    print(f'PyTorch: {torch.__version__}')
else:
    print('AVISO: Sem GPU! O treinamento sera muito lento.')
    print('Va em Runtime > Change runtime type > GPU')

In [ ]:
#@title 2. Configuracao do Modelo { run: "auto" }

MODEL_SIZE = 'medium' #@param ['small', 'medium', 'large']
DATA_MODE = 'download_mc4' #@param ['download_mc4', 'upload_existing']
DOWNLOAD_MB = 200 #@param {type: "slider", min: 50, max: 2000, step: 50}

CONFIGS = {
    'small':  {'V': 1024,  'D': 512,  'H': 8,  'FF': 2048, 'T': 512,  'N': 10,
               'B': 64, 'PT_STEPS': 10000, 'FT_STEPS': 4000,
               'PT_LR': 3e-4, 'FT_LR': 1e-4},
    'medium': {'V': 16384, 'D': 768,  'H': 12, 'FF': 3072, 'T': 1024, 'N': 12,
               'B': 32, 'PT_STEPS': 20000, 'FT_STEPS': 5000,
               'PT_LR': 3e-4, 'FT_LR': 1e-4},
    'large':  {'V': 32000, 'D': 1024, 'H': 16, 'FF': 4096, 'T': 1024, 'N': 24,
               'B': 8,  'PT_STEPS': 30000, 'FT_STEPS': 8000,
               'PT_LR': 2e-4, 'FT_LR': 5e-5},
}

cfg = CONFIGS[MODEL_SIZE]
V  = cfg['V']
D  = cfg['D']
H  = cfg['H']
HD = D // H
FF = cfg['FF']
T  = cfg['T']
N  = cfg['N']
B  = cfg['B']

# Auto-ajustar batch size pela VRAM disponivel
if device.type == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if MODEL_SIZE == 'large' and vram < 30:
        B = max(2, int(B * vram / 40))
        print(f'Batch size ajustado para {B} (VRAM={vram:.0f}GB)')
    elif MODEL_SIZE == 'medium' and vram < 12:
        B = max(4, int(B * vram / 16))
        print(f'Batch size ajustado para {B} (VRAM={vram:.0f}GB)')

n_est = V*D + T*D + N*(4*D*D + 4*D + 2*D*FF + D + FF + 4*D) + 2*D + D*V + V
print(f'''
{'='*60}
  NexaLang Transformer v8 — {MODEL_SIZE.upper()}
  Parametros: ~{n_est/1e6:.1f}M
  V={V}, D={D}, H={H}, HD={HD}, FF={FF}, T={T}, N={N}, B={B}
  Pretrain: {cfg["PT_STEPS"]} steps, LR={cfg["PT_LR"]}
  Finetune: {cfg["FT_STEPS"]} steps, LR={cfg["FT_LR"]}
{'='*60}''')

In [ ]:
#@title 3. Preparacao dos Dados

os.makedirs('data/pretrain', exist_ok=True)
os.makedirs('data/finetune', exist_ok=True)

def save_token_bin(tokens, path, vocab_size):
    """Save tokens in NexaLang binary format: [n:i32][V:i32][data:n*i32]"""
    with open(path, 'wb') as f:
        f.write(struct.pack('ii', len(tokens), vocab_size))
        f.write(np.array(tokens, dtype=np.int32).tobytes())
    print(f'  {path}: {len(tokens):,} tokens')

if DATA_MODE == 'download_mc4':
    import sentencepiece as spm
    from datasets import load_dataset

    # === Download Portuguese text from mC4 ===
    RAW_FILE = 'data/raw_pt.txt'
    target_chars = DOWNLOAD_MB * 1_000_000

    if not os.path.exists(RAW_FILE):
        print(f'Downloading ~{DOWNLOAD_MB}MB of Portuguese text from mC4...')
        collected = 0
        n_docs = 0
        with open(RAW_FILE, 'w', encoding='utf-8') as f:
            ds = load_dataset('allenai/c4', 'pt', split='train', streaming=True)
            for item in tqdm(ds, desc='Downloading', unit=' docs'):
                text = item['text'].strip()
                if len(text) < 200:
                    continue
                f.write(text + '\n')
                collected += len(text)
                n_docs += 1
                if collected >= target_chars:
                    break
        print(f'Coletados {collected/1e6:.1f}M chars em {n_docs:,} documentos')
    else:
        collected = os.path.getsize(RAW_FILE)
        print(f'Usando {RAW_FILE} existente ({collected/1e6:.1f}MB)')

    # === Train SentencePiece tokenizer ===
    SP_PREFIX = 'data/tokenizer'
    if not os.path.exists(SP_PREFIX + '.model'):
        print(f'\nTreinando tokenizer SentencePiece (V={V})...')
        spm.SentencePieceTrainer.train(
            input=RAW_FILE,
            model_prefix=SP_PREFIX,
            vocab_size=V,
            model_type='bpe',
            byte_fallback=True,
            character_coverage=0.9999,
            num_threads=os.cpu_count() or 2,
            input_sentence_size=5_000_000,
            shuffle_input_sentence=True,
            max_sentence_length=16384,
            pad_id=3,
            unk_id=0,
            bos_id=1,
            eos_id=2,
        )
        print('Tokenizer treinado!')
    sp = spm.SentencePieceProcessor(model_file=SP_PREFIX + '.model')
    actual_V = sp.get_piece_size()
    print(f'Vocab real: {actual_V}')
    if actual_V != V:
        print(f'AVISO: ajustando V de {V} para {actual_V}')
        V = actual_V

    # === Tokenize pretrain data ===
    PT_TRAIN = 'data/pretrain/train.bin'
    if not os.path.exists(PT_TRAIN):
        print('\nTokenizando corpus para pretrain...')
        all_ids = []
        with open(RAW_FILE, encoding='utf-8') as f:
            for line in tqdm(f, desc='Tokenizing', unit=' lines'):
                line = line.strip()
                if line:
                    all_ids.extend(sp.encode(line))
        all_ids = np.array(all_ids, dtype=np.int32)
        split_idx = int(len(all_ids) * 0.95)
        save_token_bin(all_ids[:split_idx], 'data/pretrain/train.bin', V)
        save_token_bin(all_ids[split_idx:], 'data/pretrain/val.bin', V)
    else:
        print('Dados pretrain ja existem.')

    # === Create instruction finetune data ===
    FT_TRAIN = 'data/finetune/train.bin'
    if not os.path.exists(FT_TRAIN):
        print('\nCriando dados de instrucao para finetune...')
        random.seed(42)
        with open(RAW_FILE, encoding='utf-8') as f:
            lines = [l.strip() for l in f if len(l.strip()) > 200]
        random.shuffle(lines)

        prompts_tpl = [
            'Resuma o seguinte texto:',
            'Continue o seguinte texto:',
            'Reescreva de forma mais clara:',
            'Qual e o tema principal deste texto?',
            'Explique o conteudo deste trecho:',
            'Faca um resumo breve:',
            'De que trata este texto?',
        ]

        ft_ids = []
        for line in tqdm(lines[:80000], desc='Gerando instrucoes'):
            words = line.split()
            if len(words) < 30:
                continue
            mid = len(words) // 2
            ctx = ' '.join(words[:mid])[:300]
            resp = ' '.join(words[mid:])[:400]
            p = random.choice(prompts_tpl)
            entry = f'[P] {p} {ctx}\n[R] {resp}\n\n'
            ft_ids.extend(sp.encode(entry))

        ft_ids = np.array(ft_ids, dtype=np.int32)
        ft_split = int(len(ft_ids) * 0.9)
        save_token_bin(ft_ids[:ft_split], 'data/finetune/train.bin', V)
        save_token_bin(ft_ids[ft_split:], 'data/finetune/val.bin', V)
    else:
        print('Dados finetune ja existem.')

    print('\nDados prontos!')

elif DATA_MODE == 'upload_existing':
    from google.colab import files
    print('Faca upload dos arquivos de dados do NexaLang:')
    print('  - text_pt_v2_bpe/train.bin (pretrain train)')
    print('  - text_pt_v2_bpe/val.bin   (pretrain val)')
    print('  - instruct_pt_v2_bpe/train.bin (finetune train, opcional)')
    print('  - instruct_pt_v2_bpe/val.bin   (finetune val, opcional)')
    print('  - bpe_merges.bin (tokenizer, opcional)')
    print()
    uploaded = files.upload()
    for name, content in uploaded.items():
        if 'bpe' in name.lower() or 'merge' in name.lower():
            with open('data/bpe_merges.bin', 'wb') as f:
                f.write(content)
            print(f'  -> data/bpe_merges.bin')
        elif 'instruct' in name.lower() or 'finetune' in name.lower():
            fn = 'train.bin' if 'train' in name else 'val.bin'
            with open(f'data/finetune/{fn}', 'wb') as f:
                f.write(content)
            print(f'  -> data/finetune/{fn}')
        else:
            fn = 'train.bin' if 'train' in name else 'val.bin'
            with open(f'data/pretrain/{fn}', 'wb') as f:
                f.write(content)
            print(f'  -> data/pretrain/{fn}')
    print('Upload concluido!')

In [ ]:
#@title 4. Definicao do Modelo (Pre-Norm Transformer)

class TransformerBlock(nn.Module):
    def __init__(self, D, H, HD, FF):
        super().__init__()
        self.ln1 = nn.LayerNorm(D)
        self.Wq = nn.Linear(D, D)
        self.Wk = nn.Linear(D, D)
        self.Wv = nn.Linear(D, D)
        self.Wo = nn.Linear(D, D)
        self.ln2 = nn.LayerNorm(D)
        self.ff1 = nn.Linear(D, FF)
        self.ff2 = nn.Linear(FF, D)
        self.H = H
        self.HD = HD

    def forward(self, x, mask):
        Bsz, Tsz, Dim = x.shape
        # Pre-norm attention
        h = self.ln1(x)
        q = self.Wq(h).view(Bsz, Tsz, self.H, self.HD).transpose(1, 2)
        k = self.Wk(h).view(Bsz, Tsz, self.H, self.HD).transpose(1, 2)
        v = self.Wv(h).view(Bsz, Tsz, self.H, self.HD).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / (self.HD ** 0.5)
        scores = scores + mask
        probs = F.softmax(scores, dim=-1)
        attn = (probs @ v).transpose(1, 2).reshape(Bsz, Tsz, Dim)
        x = x + self.Wo(attn)

        # Pre-norm FFN with GELU
        h2 = self.ln2(x)
        x = x + self.ff2(F.gelu(self.ff1(h2)))
        return x


class TransformerLM(nn.Module):
    def __init__(self, V, D, H, HD, FF, T, N):
        super().__init__()
        self.V = V
        self.D = D
        self.H = H
        self.HD = HD
        self.FF = FF
        self.T = T
        self.N = N
        self.tok_emb = nn.Embedding(V, D)
        self.pos_emb = nn.Embedding(T, D)
        self.layers = nn.ModuleList([TransformerBlock(D, H, HD, FF) for _ in range(N)])
        self.ln_f = nn.LayerNorm(D)
        self.out_proj = nn.Linear(D, V)

    def forward(self, tokens):
        Bsz, Tsz = tokens.shape
        pos = torch.arange(Tsz, device=tokens.device)
        x = self.tok_emb(tokens) + self.pos_emb(pos)

        mask = torch.triu(
            torch.full((Tsz, Tsz), float('-inf'), device=tokens.device),
            diagonal=1
        ).unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, mask)

        return self.out_proj(self.ln_f(x))


model = TransformerLM(V, D, H, HD, FF, T, N).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Modelo criado: {n_params:,} parametros ({n_params*4/1e9:.2f} GB fp32)')
print(f'Arquitetura: Pre-Norm Transformer ({N} layers, {H} heads)')

In [ ]:
#@title 5. Funcoes de Treinamento

def load_tokens(path):
    """Load BPE tokens from NexaLang binary format."""
    with open(path, 'rb') as f:
        n_tokens = struct.unpack('i', f.read(4))[0]
        vocab_size = struct.unpack('i', f.read(4))[0]
        data = np.frombuffer(f.read(n_tokens * 4), dtype=np.int32).copy()
    print(f'  {path}: {n_tokens:,} tokens (V={vocab_size})')
    return data


def get_batch(tokens, batch_size, seq_len):
    """Sample random batch of (input, target) pairs."""
    n = len(tokens) - seq_len - 1
    starts = np.random.randint(0, n, size=batch_size)
    x = np.stack([tokens[s:s+seq_len] for s in starts])
    y = np.stack([tokens[s+1:s+seq_len+1] for s in starts])
    return (torch.tensor(x, dtype=torch.long, device=device),
            torch.tensor(y, dtype=torch.long, device=device))


def get_lr(step, max_lr, min_lr, warmup, total_steps):
    """Cosine schedule with linear warmup."""
    if step < warmup:
        return max_lr * step / max(1, warmup)
    progress = (step - warmup) / max(1, total_steps - warmup)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


def save_weights_nxl(model, path):
    """Save weights in NexaLang-compatible binary format.
    PyTorch Linear: [out, in] -> NexaLang: [in, out] (transpose)."""
    sd = model.state_dict()
    V = model.V
    D = model.D
    H = model.H
    FF = model.FF
    T = model.T
    N = model.N

    with open(path, 'wb') as f:
        # Header
        f.write(struct.pack('iiiiii', V, D, H, FF, T, N))

        def w(key):
            f.write(sd[key].float().cpu().numpy().tobytes())

        def wt(key):
            f.write(sd[key].float().cpu().numpy().T.copy().tobytes())

        # Embeddings
        w('tok_emb.weight')
        w('pos_emb.weight')

        # Per-weight-type order (NexaLang load order)
        weight_specs = [
            ('Wq.weight', True),  ('Wq.bias', False),
            ('Wk.weight', True),  ('Wk.bias', False),
            ('Wv.weight', True),  ('Wv.bias', False),
            ('Wo.weight', True),  ('Wo.bias', False),
            ('ln1.weight', False),('ln1.bias', False),
            ('ff1.weight', True), ('ff1.bias', False),
            ('ff2.weight', True), ('ff2.bias', False),
            ('ln2.weight', False),('ln2.bias', False),
        ]
        for suffix, transpose in weight_specs:
            for i in range(N):
                key = f'layers.{i}.{suffix}'
                if transpose:
                    wt(key)
                else:
                    w(key)

        # Final layer norm + output projection
        w('ln_f.weight')
        w('ln_f.bias')
        wt('out_proj.weight')
        w('out_proj.bias')

    sz = os.path.getsize(path) / 1e6
    print(f'  Pesos salvos em {path} ({sz:.1f} MB)')


def train_model(model, mode='pretrain'):
    """Main training loop with mixed precision."""
    is_pt = mode == 'pretrain'

    if is_pt:
        train_data = load_tokens('data/pretrain/train.bin')
        val_data = load_tokens('data/pretrain/val.bin')
        steps = cfg['PT_STEPS']
        max_lr = cfg['PT_LR']
        warmup = 500
        out_path = 'model_v8.bin'
    else:
        train_data = load_tokens('data/finetune/train.bin')
        val_data = load_tokens('data/finetune/val.bin')
        steps = cfg['FT_STEPS']
        max_lr = cfg['FT_LR']
        warmup = 200
        out_path = 'model_v8_ft.bin'

    min_lr = max_lr / 10
    eval_int = max(steps // 50, 100)

    print(f'\n{"="*60}')
    print(f'  {"PRETRAIN" if is_pt else "FINETUNE"} — {n_params/1e6:.1f}M params')
    print(f'  {len(train_data):,} train / {len(val_data):,} val tokens')
    print(f'  {steps} steps, B={B}, T={T}, LR={max_lr}')
    print(f'{"="*60}\n')

    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler()

    model.train()
    t0 = time.time()
    best_val = float('inf')
    log_losses = []

    for step in range(steps):
        lr = get_lr(step, max_lr, min_lr, warmup, steps)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        inputs, targets = get_batch(train_data, B, T)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            logits = model(inputs)
            loss = F.cross_entropy(logits.view(-1, V), targets.view(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        if step % eval_int == 0:
            model.eval()
            with torch.no_grad():
                val_losses = []
                for _ in range(5):
                    vi, vt = get_batch(val_data, B, T)
                    with torch.cuda.amp.autocast():
                        vl = F.cross_entropy(model(vi).view(-1, V), vt.view(-1))
                    val_losses.append(vl.item())
            val_loss = sum(val_losses) / len(val_losses)
            elapsed = time.time() - t0
            tok_s = (step + 1) * B * T / elapsed if elapsed > 0 else 0

            print(f'  Step {step:6d}/{steps}  loss={loss.item():.4f}  '
                  f'val={val_loss:.4f}  lr={lr:.6f}  '
                  f'{tok_s:.0f} tok/s  {elapsed:.0f}s')

            log_losses.append((step, loss.item(), val_loss))
            if val_loss < best_val:
                best_val = val_loss
            model.train()

    elapsed = time.time() - t0
    print(f'\n  Done! {elapsed:.0f}s ({elapsed/60:.1f} min)')
    print(f'  Best val loss: {best_val:.4f}')

    # Save weights
    save_weights_nxl(model, out_path)

    # Also save PyTorch checkpoint
    torch.save({
        'model': model.state_dict(),
        'config': {'V': V, 'D': D, 'H': H, 'HD': HD, 'FF': FF, 'T': T, 'N': N},
        'best_val_loss': best_val,
        'steps': steps,
        'log': log_losses,
    }, out_path.replace('.bin', '_torch.pt'))

    return model, log_losses

print('Funcoes de treinamento prontas.')

In [ ]:
#@title 6. Executar Pretrain

model, pt_log = train_model(model, 'pretrain')

In [ ]:
#@title 7. Executar Finetune (opcional)

if os.path.exists('data/finetune/train.bin'):
    model, ft_log = train_model(model, 'finetune')
else:
    print('Sem dados de finetune. Pulando.')
    ft_log = []

In [ ]:
#@title 8. Gerar Texto de Exemplo

def generate_text(model, prompt, max_tokens=200, temperature=0.7):
    """Generate text using the trained model."""
    model.eval()

    # Encode prompt
    if DATA_MODE == 'download_mc4':
        import sentencepiece as spm
        sp = spm.SentencePieceProcessor(model_file='data/tokenizer.model')
        prompt_ids = sp.encode(prompt)

        # Build decode table
        def decode_ids(ids):
            return sp.decode(ids)
    else:
        # Byte-level fallback for uploaded BPE data
        prompt_ids = list(prompt.encode('utf-8'))
        def decode_ids(ids):
            return bytes(min(b, 255) for b in ids).decode('utf-8', errors='replace')

    # Pad to context length
    if len(prompt_ids) > T:
        prompt_ids = prompt_ids[-T:]
    tokens = [0] * (T - len(prompt_ids)) + prompt_ids

    gen_ids = []
    with torch.no_grad():
        for _ in range(max_tokens):
            ctx = torch.tensor([tokens], dtype=torch.long, device=device)
            logits = model(ctx)
            last_logits = logits[0, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_tok = torch.multinomial(probs, 1).item()
            gen_ids.append(next_tok)
            tokens = tokens[1:] + [next_tok]

            # Stop on double newline
            if DATA_MODE == 'download_mc4':
                partial = sp.decode(gen_ids[-5:])
                if '\n\n' in partial and len(gen_ids) > 10:
                    break

    return decode_ids(gen_ids)


# Test prompts
prompts = [
    '[P] Ola! Como voce esta?\n[R] ',
    '[P] O que e inteligencia artificial?\n[R] ',
    '[P] Explique o que e o Brasil.\n[R] ',
]

print('=' * 60)
print('  Exemplos de geracao de texto')
print('=' * 60)
for prompt in prompts:
    print(f'\nPrompt: {prompt.strip()}')
    text = generate_text(model, prompt)
    print(f'Resposta: {text}')
    print('-' * 40)

In [ ]:
#@title 9. Exportar Tokenizer para NexaLang

def export_tokenizer_nxl(sp_model_path, out_path):
    """Convert SentencePiece model to NexaLang bpe_merges.bin format.

    Format:
      [num_merges: i32] [vocab_size: i32] [max_tok_len: i32]
      [merge_a: num_merges * i32]
      [merge_b: num_merges * i32]
      [decode_len: vocab_size * i32]
      [decode_data: vocab_size * max_tok_len * u8]
    """
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor(model_file=sp_model_path)
    vocab_size = sp.get_piece_size()

    # === Build decode table ===
    decode_bytes = {}
    max_tok_len = 1
    for i in range(vocab_size):
        piece = sp.id_to_piece(i)
        if piece.startswith('<0x') and piece.endswith('>'):
            byte_val = int(piece[3:-1], 16)
            decode_bytes[i] = bytes([byte_val])
        elif piece in ('<unk>', '<s>', '</s>', '<pad>'):
            decode_bytes[i] = b''
        else:
            text = piece.replace('\xe2\x96\x81', ' ')  # sentencepiece word boundary
            decode_bytes[i] = text.encode('utf-8')
        max_tok_len = max(max_tok_len, len(decode_bytes[i]))

    # === Extract BPE merge rules ===
    piece_to_id = {}
    for i in range(vocab_size):
        piece_to_id[sp.id_to_piece(i)] = i

    # Find where byte tokens start
    byte_start = None
    for i in range(vocab_size):
        if sp.id_to_piece(i) == '<0x00>':
            byte_start = i
            break
    merge_start = (byte_start or 0) + 256

    merge_a = []
    merge_b = []
    for pid in range(merge_start, vocab_size):
        piece = sp.id_to_piece(pid)
        found = False
        for split_pos in range(1, len(piece)):
            left = piece[:split_pos]
            right = piece[split_pos:]
            if left in piece_to_id and right in piece_to_id:
                lid = piece_to_id[left]
                rid = piece_to_id[right]
                if lid < pid and rid < pid:
                    merge_a.append(lid)
                    merge_b.append(rid)
                    found = True
                    break
        if not found:
            merge_a.append(0)
            merge_b.append(0)

    num_merges = len(merge_a)

    # === Write binary ===
    with open(out_path, 'wb') as f:
        f.write(struct.pack('iii', num_merges, vocab_size, max_tok_len))
        f.write(np.array(merge_a, dtype=np.int32).tobytes())
        f.write(np.array(merge_b, dtype=np.int32).tobytes())

        decode_len = np.zeros(vocab_size, dtype=np.int32)
        decode_data = np.zeros((vocab_size, max_tok_len), dtype=np.uint8)
        for i in range(vocab_size):
            b = decode_bytes.get(i, b'')
            decode_len[i] = len(b)
            for j, byte_val in enumerate(b):
                decode_data[i, j] = byte_val

        f.write(decode_len.tobytes())
        f.write(decode_data.tobytes())

    sz = os.path.getsize(out_path) / 1e6
    print(f'Tokenizer salvo: {out_path} ({sz:.1f} MB, V={vocab_size}, {num_merges} merges)')


if DATA_MODE == 'download_mc4':
    export_tokenizer_nxl('data/tokenizer.model', 'bpe_merges_v8.bin')
    print('\nArquivos do tokenizer:')
    print('  - bpe_merges_v8.bin    (formato NexaLang)')
    print('  - data/tokenizer.model (SentencePiece original)')
else:
    print('Modo upload: usando tokenizer existente (bpe_merges.bin)')

In [ ]:
#@title 10. Baixar Arquivos

from google.colab import files as colab_files

download_list = []
for f in ['model_v8.bin', 'model_v8_ft.bin', 'bpe_merges_v8.bin',
          'data/tokenizer.model', 'model_v8_torch.pt', 'model_v8_ft_torch.pt']:
    if os.path.exists(f):
        sz = os.path.getsize(f) / 1e6
        download_list.append((f, sz))
        print(f'  {f}: {sz:.1f} MB')

print(f'\nTotal: {sum(s for _, s in download_list):.1f} MB')
print('\nBaixando arquivos...')
for f, _ in download_list:
    try:
        colab_files.download(f)
    except Exception as e:
        print(f'  Erro ao baixar {f}: {e}')

print('\n' + '='*60)
print('  CONCLUIDO!')
print('='*60)
print(f'''
Para usar no NexaLang:
  1. Copie model_v8.bin e model_v8_ft.bin para o diretorio do NexaLang
  2. Copie bpe_merges_v8.bin para data/bpe_merges.bin
  3. Atualize o chat NexaLang para V={V}, D={D}, H={H}, FF={FF}, T={T}, N={N}
  4. Recompile e execute!
''')

---

## Notas

### Arquitetura
- **Pre-norm Transformer** (LayerNorm antes de attention/FFN)
- Mesma arquitetura do NexaLang v7+
- GELU activation no FFN
- Mixed precision (fp16) no treinamento

### Formato dos pesos
- Header: `[V, D, H, FF, T, N]` (6 × int32)
- `tok_emb [V×D]`, `pos_emb [T×D]`
- Per-weight-type: todos Wq de todas as layers, depois todos bq, etc.
- Linear weights transpostos: PyTorch `[out,in]` → NexaLang `[in,out]`
- `ln_f`, `out_proj` no final

### GPU recomendada
| Modelo | GPU minima | Tempo estimado |
|--------|-----------|----------------|
| small (33M) | T4 16GB | ~30 min |
| medium (150M) | T4 16GB | ~2-4 horas |
| large (500M) | A100 40GB | ~4-8 horas |